GOAL: Python script to make my workouts for me

FUNCTION:
- Input workout (swim/gym)
- Auto generate workout
- Gym: match 3 different sets for each type of rotation, with mixed difficulty
- Swim: group sets based on focus

ACTIONS:
1) Write all the sets into an excel sheet
2) Write script to read in excel
3) Group sets
4) Auto generate workout, time limit/yardage
5) Make sure you put in a cycle restraint (cannot have the same set within the span of ~2 weeks)

SWIM:
- Pull 3-4 warmup
- Pull 2 from preset fragmented, or 1 from preset full
- Group mains by type so that you can only have one from each type, specifically for fragments --> mitigate repeats

In [ ]:
import pandas as pd 
import random
from datetime import date

In [2]:
filename = "/Users/iec2113/Documents/iec_work.xlsx"
gym_opts = pd.read_excel(filename, sheet_name="Gym")
swim_opts = pd.read_excel(filename, sheet_name="Swim")

GYM WORKOUT

In [36]:
def create_gym_workout(gym_opts):
    def grouping(gym_opts):
        grouped_by_level = gym_opts.groupby("Level")

        warmup = grouped_by_level.get_group("Warmup")
        core = grouped_by_level.get_group("Core")
        arms = grouped_by_level.get_group("Arms")
        legs = grouped_by_level.get_group("Legs")

        return warmup, core, arms, legs
    warmup, core, arms, legs = grouping(gym_opts)



    def pick_gym_workout(warmup, core, arms, legs):
        random_warmup = warmup.sample(2)
        random_core = core.sample(3)
        random_arms = arms.sample(3)
        random_legs = legs.sample(3)

        gym_workout = pd.concat([random_warmup, random_core, random_arms, random_legs])
        gym_workout["Reps"] = gym_workout.apply(lambda row: random.randint(int(row["RandInt Min"]), int(row["RandInt Max"])), axis=1)

        return gym_workout
    gym_workout = pick_gym_workout(warmup, core, arms, legs)



    def print_workout(gym_workout):
        level_order = ["Warmup", "Core", "Arms", "Legs"]
        sets = {"Warmup": 2, "Core": 3, "Arms": 3, "Legs": 3}

        for level in level_order:
            group = gym_workout[gym_workout["Level"] == level]

            print(f"\n{'='*20} {level.upper()} {'='*20}\n")

            exercises = list(group.iterrows())
            mid = len(exercises) // 2
            num_sets = sets[level]

            print("     —")
            for i, (_, row) in enumerate(exercises):
                reps = row["Reps"]
                exercise = row["Set"]
                if i == mid:
                    print(f"{num_sets}x  |  {reps} {exercise}")
                else:
                    print(f"    |  {reps} {exercise}")
            print("     —\n")
    print_workout(gym_workout)
create_gym_workout(gym_opts)



==================== WARMUP ====================

     —
    |  6 ea plank pos swimmers
2x  |  13 band pull aparts
     —


==================== CORE ====================

     —
    |  5 renegade rows
3x  |  9 push up to plank position
    |  8 ea windshield wipers
     —


==================== ARMS ====================

     —
    |  6 ea standing dumbbell push press
3x  |  5 ea arm circles
    |  14 laying dumbbell push press
     —


==================== LEGS ====================

     —
    |  15 ea weighted box stepups
3x  |  20 medball squat throws
    |  5 front rack lunge
     —



SWIM WORKOUT

In [ ]:
def create_swim_workout(swim_opts):
    def grouping(swim_opts):
        grouped_by_level = swim_opts.groupby("Level")

        warmup = grouped_by_level.get_group("Warmup")
        grouped_by_type = warmup.groupby("Workout Type")
        swim_warmup = grouped_by_type.get_group("Swim")
        kick_warmup = grouped_by_type.get_group("Kick")
        pull_warmup = grouped_by_type.get_group("Pull")
        var_warmup = grouped_by_type.get_group("Variable")

        preset = grouped_by_level.get_group("Preset")
        main = grouped_by_level.get_group("Main")
        warmdown = grouped_by_level.get_group("Warmdown")

        return swim_warmup, kick_warmup, pull_warmup, var_warmup, preset, main, warmdown
    swim_warmup, kick_warmup, pull_warmup, var_warmup, preset, main, warmdown = grouping(swim_opts)


    def swim_workout(swim_warmup, kick_warmup, pull_warmup, var_warmup, preset, main, warmdown):
        random_swim_warmup = swim_warmup.sample(1)
        random_kick_warmup = kick_warmup.sample(1)
        random_pull_warmup = pull_warmup.sample(1)
        random_var_warmup = var_warmup.sample(1)
        random_preset = preset.sample(1)
        random_main = main.sample(1)
        random_warmdown = warmdown.sample(1)

        swim_workout = pd.concat([random_swim_warmup, random_kick_warmup, random_pull_warmup, 
                                    random_var_warmup, random_preset, random_main, random_warmdown])
        return swim_workout

    swim_workout = swim_workout(swim_warmup,kick_warmup, pull_warmup, var_warmup, preset, main, warmdown)




    def print_workout(swim_workout):
        level_order = ["Warmup", "Preset", "Main", "Warmdown"]
        today = date.today()
        chill = today.strftime("%A, %B %d, %Y")
        print(f"\n{'='*20} Swim {chill} {'='*20}\n")

        for level in level_order:
            group = swim_workout[swim_workout["Level"] == level]
            exercises = list(group.iterrows())

        
            for _, row in exercises:
                exercise = str(row["Set"]).replace("\\n", "\n")
                rounds = int(row["Rounds"])
                lines = exercise.strip().split("\n")
                mid = len(lines) // 2
                if rounds != 1:
                    print("     —")
                    for i, line in enumerate(lines):
                        if i == mid:
                            print(f"{rounds}x  | {line}")
                        else:
                            print(f"    | {line}")
                    print("     —\n")
                else: 
                    print(f"{exercise}")
            print("\n")

    print_workout = print_workout(swim_workout)
create_swim_workout(swim_opts)


==================== Swim Saturday, June 06, 2026 ====================

200 sw
200 IM
400 pull
6x50 rot IM


200 THINK


10x100 rest :10


2x100 ascend 1:40


